In [3]:
#Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [4]:
#Import Database / Data

In [5]:
conn = sqlite3.connect('customer_churn.db')

#Standard SQL query that will pick up the names from the table
sql_query = '''
        SELECT name
        FROM sqlite_master
        WHERE type='table'
'''

tables = pd.read_sql(sql_query, conn)

#Create dataframe for each table
for table_name in tables['name']:
    df = pd.read_sql("SELECT * FROM {}".format(table_name), conn)
    globals()[f"df_{table_name}"] = df
    print(f"Created dataframe: df_{table_name}")

#Closing the connection
conn.close()

Created dataframe: df_db_customer
Created dataframe: df_db_subscription
Created dataframe: df_db_support


In [6]:
# Print table names and column names
conn = sqlite3.connect('customer_churn.db')

for table_name in tables['name']:
    print(f"\nTable Name: {table_name}")
    # Get column information
    columns_query = f"PRAGMA table_info({table_name});"
    columns = pd.read_sql(columns_query, conn)
    print("Columns:")
    print(columns['name'].tolist())

# Close connection
conn.close()


Table Name: db_customer
Columns:
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

Table Name: db_subscription
Columns:
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

Table Name: db_support
Columns:
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']


In [7]:
#Data Cleaning

In [8]:
#df_db_customer.head()

df_db_customer.tail()

,customerid,name,country,state,gender,dob,interests,pincode
16,0020-JDNXP,rikim,India,Meghalaya,Female,1994-08-19 00:00:00,NaN,None
17,0021-IKXGC,vishakha,India,Rajasthan,Female,2000-09-02 00:00:00,NaN,None
18,0022-TCJCI,raghvendra,India,Telangana,Male,1983-12-30 00:00:00,NaN,None
19,0023-HGHWL,rishabh,India,Uttar Pradesh,Men,1991-05-14 00:00:00,NaN,None
20,0023-UYUPN,sudevi,India,Maharashtra,Women,1977-10-06 00:00:00,NaN,None


In [9]:
df_db_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerid  21 non-null     str   
 1   name        21 non-null     str   
 2   country     18 non-null     str   
 3   state       21 non-null     str   
 4   gender      21 non-null     str   
 5   dob         21 non-null     str   
 6   interests   4 non-null      str   
 7   pincode     0 non-null      object
dtypes: object(1), str(7)
memory usage: 1.4+ KB


In [10]:
#1. Rename name to customer_name
#2. Drop columns = Interests & Pincode
#3. Change dataType of DOB
#4. Data Standardization
#5. Fix Missing Values of countries

In [11]:
#1. Rename name to customer_name

df_db_customer.rename(columns = {'name' : 'customer_name'}, inplace= True)

In [12]:
#2. Drop columns = Interests & Pincode
# errors='ignore' makes this safe to re-run even if the columns are already gone

df_db_customer.drop(columns=['interests', 'pincode'], inplace=True, errors='ignore')

In [13]:
df_db_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   customerid     21 non-null     str  
 1   customer_name  21 non-null     str  
 2   country        18 non-null     str  
 3   state          21 non-null     str  
 4   gender         21 non-null     str  
 5   dob            21 non-null     str  
dtypes: str(6)
memory usage: 1.1 KB


In [14]:
#3. Change dataType of DOB

pd.to_datetime(df_db_customer['dob'])

0    1982-04-12
1    1995-11-23
2    1978-02-15
3    2001-08-30
4    1990-05-05
5    1988-12-10
6    1976-09-21
7    1999-03-14
8    1985-07-07
9    1993-10-29
10   1997-01-22
11   1981-06-18
12   2004-12-01
13   1992-04-25
14   1979-11-11
15   1986-02-28
16   1994-08-19
17   2000-09-02
18   1983-12-30
19   1991-05-14
20   1977-10-06
Name: dob, dtype: datetime64[us]

In [15]:
#4. Data Standardization

#df_db_customer['gender'].unique()
df_db_customer['gender'].rename({'Men' : 'Male', 'Women' : 'Female'}, inplace= True)

In [16]:
#5. Fix Missing Values of countries

df_db_customer[df_db_customer['country'].isna()]

,customerid,customer_name,country,state,gender,dob
5,0013-MHZWF,durga,NaN,Delhi,Women,1988-12-10 00:00:00
8,0015-UOCOJ,maya,NaN,Kathmandu,Women,1985-07-07 00:00:00
12,0018-NYROU,chitra,NaN,Telangana,Female,2004-12-01 00:00:00


In [17]:
#Country and State = Unique value pair

state_country_mapping = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()

df_db_customer['country'] = df_db_customer['country'].fillna(df_db_customer['state'].map(state_country_mapping))

In [18]:
#df_db_subscription.head()
df_db_subscription.tail()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
16,0020-JDNXP,2022-01-19,Organic,2024-01-19,Premium,Annual,NaN,NaN,20.99,550,62
17,0021-IKXGC,2021-07-07,Paid,2025-07-07,Standard,Annual,NaN,NaN,13.99,840,27
18,0022-TCJCI,2023-09-14,Refferal,2024-09-14,Basic,Monthly,2024-09-14,Forgot to cancel trial,16.99,42,99
19,0023-HGHWL,2020-06-23,Organic,2025-06-23,Premium,Annual,NaN,NaN,22.99,1955,7
20,0023-UYUPN,2022-12-31,Paid,2025-12-31,Standard,Monthly,NaN,NaN,13.99,790,47


In [19]:
df_db_subscription.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               21 non-null     str    
 1   subscription_start_date  21 non-null     str    
 2   subscription_type        21 non-null     str    
 3   renewal_date             21 non-null     str    
 4   plan_type                21 non-null     str    
 5   contract_type            21 non-null     str    
 6   cancellation_date        6 non-null      str    
 7   cancellation_reason      6 non-null      str    
 8   monthly_charges          21 non-null     float64
 9   cltv                     21 non-null     int64  
 10  churn_score              21 non-null     int64  
dtypes: float64(1), int64(2), str(8)
memory usage: 1.9 KB


In [20]:
date_col = ['subscription_start_date', 'renewal_date', 'cancellation_date']
df_db_subscription[date_col].apply(pd.to_datetime)

,subscription_start_date,renewal_date,cancellation_date
0,2021-03-15,2025-03-15,NaT
1,2020-08-01,2024-08-01,2024-09-10
2,2022-11-20,2025-11-20,NaT
3,2019-05-10,2025-05-10,NaT
4,2023-01-05,2024-01-05,2024-02-28
5,2022-06-18,2025-06-18,NaT
6,2021-09-30,2024-09-30,2024-11-15
7,2020-02-14,2025-02-14,NaT
8,2023-07-22,2024-07-22,NaT
9,2022-04-03,2025-04-03,NaT


In [21]:
df_db_support.head()

,customerid,complaint_date,escalations,csat_score,col_1,comment
0,0003-MKNFE,2024-08-28 00:00:00,N,60,None,service issue
1,0003-MKNFE,2024-08-28 00:00:00,Y,10,None,demaned refund
2,0013-EXCHZ,2024-01-20 00:00:00,Y,20,None,NaN
3,0013-MHZWF,2025-03-18 00:00:00,N,90,None,guidance to renew
4,0013-SMEOE,2024-11-01 00:00:00,N,30,None,NaN


In [22]:
df_db_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customerid      9 non-null      str   
 1   complaint_date  9 non-null      str   
 2   escalations     9 non-null      str   
 3   csat_score      9 non-null      int64 
 4   col_1           0 non-null      object
 5   comment         4 non-null      str   
dtypes: int64(1), object(1), str(4)
memory usage: 564.0+ bytes


In [23]:
df_db_support.drop(columns=['col_1', 'comment'], inplace=True, errors='ignore')

In [24]:
df_db_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   customerid      9 non-null      str  
 1   complaint_date  9 non-null      str  
 2   escalations     9 non-null      str  
 3   csat_score      9 non-null      int64
dtypes: int64(1), str(3)
memory usage: 420.0 bytes


In [25]:
df_db_support['complaint_count'] = df_db_support.groupby('customerid')['customerid'].transform('count')

In [26]:
df_db_support = df_db_support.sort_values('complaint_date').drop_duplicates('customerid', keep= 'last')

In [27]:
#Feature Engineering and Data Analysis

In [28]:
df_db_subscription['churn_flag'] = np.where(df_db_subscription['cancellation_date'].notna(), 1, 0)

In [29]:
df = (df_db_subscription
        .merge(df_db_customer, on='customerid', how='left')
        .merge(df_db_support, on='customerid', how='left'))

In [30]:
df.shape

(21, 21)

In [31]:
df.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,...,churn_flag,customer_name,country,state,gender,dob,complaint_date,escalations,csat_score,complaint_count
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaN,NaN,13.99,627,...,0,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,NaN,NaN,NaN,NaN
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,...,1,raghav,India,Karnataka,Male,1995-11-23 00:00:00,2024-08-28 00:00:00,Y,10.0,2.0
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaN,NaN,6.99,210,...,0,lalita,India,Delhi,Female,1978-02-15 00:00:00,NaN,NaN,NaN,NaN
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaN,NaN,22.99,1725,...,0,mohan,India,Nagaland,Male,2001-08-30 00:00:00,NaN,NaN,NaN,NaN
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,...,1,mira,India,Delhi,Female,1990-05-05 00:00:00,2024-01-20 00:00:00,Y,20.0,1.0


In [32]:
df_db_support.head()

,customerid,complaint_date,escalations,csat_score,complaint_count
2,0013-EXCHZ,2024-01-20 00:00:00,Y,20,1
5,0017-IUDMW,2024-04-10 00:00:00,Y,25,1
1,0003-MKNFE,2024-08-28 00:00:00,Y,10,2
8,0022-TCJCI,2024-09-14 00:00:00,N,90,2
6,0019-EFAEP,2024-09-27 00:00:00,Y,30,1


In [33]:
df.to_csv('exported_churn_data.csv', index=False)

In [34]:
#Data Analysis

In [35]:
#1. Churn Rate

df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'complaint_date', 'escalations', 'csat_score', 'complaint_count'],
      dtype='str')

In [36]:
churn_rate = df['churn_flag'].mean()*100
print("Churn rate = ", round(churn_rate, 2), "%")

Churn rate =  28.57 %


In [37]:
#2. Retention Rate

retention_rate = 100 - churn_rate
print("Retention rate = ", round(retention_rate, 2), "%")

Retention rate =  71.43 %


In [38]:
#3. Churn by plan type

churn_by_plan = (df.groupby('plan_type')['churn_flag'].mean().mul(100).round(2).reset_index(name= 'churn_rate_pct'))
print(churn_by_plan)

  plan_type  churn_rate_pct
0     Basic           60.00
1   Premium           14.29
2  Standard           22.22


In [39]:
#4.A Churn by State + Sum(Revenue) & Count of Users

churn_by_state = df.groupby('state')[['churn_flag', 'monthly_charges']].agg({'churn_flag': ['mean', 'count'], 'monthly_charges': 'sum'}).reset_index()
churn_by_state.columns = ['state', 'churn_rate_pct', 'user_count', 'total_revenue']
churn_by_state['churn_rate_pct'] = (churn_by_state['churn_rate_pct'] * 100).round(2)

print(churn_by_state)

           state  churn_rate_pct  user_count  total_revenue
0          Delhi           25.00           4          52.96
1      Karnataka          100.00           2          20.98
2      Kathmandu            0.00           2          20.98
3    Maharashtra            0.00           3          50.97
4      Meghalaya           66.67           3          42.97
5       Nagaland            0.00           1          22.99
6      Rajasthan            0.00           2          36.98
7      Telangana           50.00           2          30.98
8  Uttar Pradesh            0.00           2         115.98


In [40]:
#4.B Churn by subscription type + Sum(Revenue) & Count of users

churn_by_subscription = df.groupby('subscription_type')[['churn_flag', 'monthly_charges']].agg({'churn_flag' : ['mean', 'count'], 'monthly_charges' : 'sum'}).reset_index()
churn_by_subscription.columns = ['subscription_type', 'churn_rate_pct', 'user_count', 'total_revenue']
churn_by_subscription['churn_rate_pct'] = (churn_by_subscription['churn_rate_pct'] * 100).round(2)

print(churn_by_subscription)

  subscription_type  churn_rate_pct  user_count  total_revenue
0           Organic            0.00           9         145.91
1              Paid           16.67           6         174.94
2          Refferal           83.33           6          74.94


In [41]:
#5 ARPU = Average Revenue Per User

arpu = df['monthly_charges'].mean()
print("ARPU = ", round(arpu, 2), "%")

ARPU =  18.85 %


In [42]:
#6 Average Customer Tenure
df['cancellation_date'] = pd.to_datetime(df['cancellation_date'])
df['subscription_start_date'] = pd.to_datetime(df['subscription_start_date'])

today = pd.Timestamp.today()

df['tenure_days'] = np.where(
            df['cancellation_date'].notna(),

    (df['cancellation_date'] - df['subscription_start_date']).dt.days,

    (today - df['subscription_start_date']).dt.days
)

avg_tenure = df['tenure_days'].mean()
print("Average Tenure = ", round(avg_tenure))

Average Tenure =  1492


In [43]:
#7. Revenue at risk (Revenue lost from losing user's)

revenue_at_risk = df.loc[df['churn_flag'] == 1, 'monthly_charges'].sum()
print("Revenue at risk = ", round(revenue_at_risk))

Revenue at risk =  74


In [44]:
#8. Escalation Rate

escalation_rate = (df['escalations']=='Y').mean()*100
print("Escalation rate = ", round(escalation_rate, 2), "%")

Escalation rate =  19.05 %


In [45]:
#9. Average Complaint by User

avg_complaints = df['complaint_count'].sum() / df['customerid'].nunique()
print("Average complaints = ", round(avg_complaints))

Average complaints =  0


In [46]:
df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'complaint_date', 'escalations', 'csat_score', 'complaint_count',
       'tenure_days'],
      dtype='str')